# Dataset B — Final six-model review

## tl;dr

Notebook này là cổng chỉ đọc cho so sánh cuối LR, RF, XGBoost, PlainMLP, RealMLP và TabMmini. Cross-model inputs chỉ được chấp nhận khi cả sáu reporting summaries đều `COMPLETE`, dùng final 15-feature contract, cùng `split_v1`, cùng ordered Test IDs và cùng bootstrap-index lineage. Hiện cả 6/6 final reporting packages đã được nối và đủ điều kiện.

## Context & Methods

Mỗi final model run sẽ xuất reporting package chuẩn gồm `model_summary.json`, `cv_fold_metrics.csv`, `rq1_metrics_long.csv`, `confusion_matrix.csv`, `calibration_curve.csv` và `artifact_index.json`. Notebook tổng hợp không huấn luyện, inference, chọn threshold hay tính lại bootstrap.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config/dataset_b/model_protocol_v1.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate the Dataset B project root.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_b.reporting import EXPECTED_REPORTING_FILES, FINAL_MODEL_KEYS, validate_cross_model_summaries, validate_reporting_directory

MODEL_SUMMARY_PATHS = {model: None for model in FINAL_MODEL_KEYS}
MODEL_SUMMARY_PATHS['lr'] = 'artifacts/dataset_b/lr/final/lr_final_15f_20261002_01/reporting/model_summary.json'
MODEL_SUMMARY_PATHS['rf'] = 'artifacts/dataset_b/rf/final/rf_final_15f_20261002_01/reporting/model_summary.json'
MODEL_SUMMARY_PATHS['xgboost'] = 'artifacts/dataset_b/xgboost/final/xgb_final_15f_20261003_01/reporting/model_summary.json'
MODEL_SUMMARY_PATHS['plain_mlp'] = 'artifacts/dataset_b/plain_mlp/final/plain_mlp_final_15f_20261004_03/reporting/model_summary.json'
MODEL_SUMMARY_PATHS['realmlp'] = 'artifacts/dataset_b/realmlp/final/realmlp_final_15f_20261004_01/reporting/model_summary.json'
MODEL_SUMMARY_PATHS['tabmmini'] = 'artifacts/dataset_b/tabmmini/final/tabmmini_final_15f_20261004_01/reporting/model_summary.json'
print('Reporting contract files:', ', '.join(EXPECTED_REPORTING_FILES))

W1005 00:22:03.450000 8104 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


C:\Users\Dan\anaconda3\envs\model_train\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Reporting contract files: model_summary.json, cv_fold_metrics.csv, rq1_metrics_long.csv, confusion_matrix.csv, calibration_curve.csv, artifact_index.json


### Key Assumptions

- Chỉ đường dẫn tới `final/<run_id>/reporting/model_summary.json` được phép điền.
- Working, legacy 12-feature và incomplete summaries không đủ điều kiện so sánh.
- Readiness kiểm tra `reporting_status` cùng publication marker (`FROZEN` hoặc `COMPLETED`). Nếu có `final_review.json` riêng, trạng thái direct review được hiển thị độc lập.
- Trường generation-time bên trong từng model summary không thay thế publication marker hoặc separate direct review.
- CI riêng từng model không tự động chứng minh chênh lệch giữa model có ý nghĩa thống kê.

## Data

### Reporting readiness

In [2]:
summaries = []
readiness_rows = []
for model_key in FINAL_MODEL_KEYS:
    configured = MODEL_SUMMARY_PATHS[model_key]
    if configured is None:
        readiness_rows.append({'model_key': model_key, 'reporting_summary': 'NOT_CONFIGURED', 'reporting_status': 'PENDING', 'publication_marker': 'MISSING', 'separate_direct_review': 'NOT_AVAILABLE', 'eligible': False})
        continue
    path = PROJECT_ROOT / configured
    if not path.is_file():
        readiness_rows.append({'model_key': model_key, 'reporting_summary': str(configured), 'reporting_status': 'MISSING', 'publication_marker': 'MISSING', 'separate_direct_review': 'NOT_AVAILABLE', 'eligible': False})
        continue
    summary = validate_reporting_directory(path.parent)
    run_root = path.parent.parent
    publication_marker = 'FROZEN' if (run_root / 'FROZEN').is_file() else ('COMPLETED' if (run_root / 'COMPLETED').is_file() else 'MISSING')
    direct_review_path = run_root / 'final_review.json'
    separate_direct_review = json.loads(direct_review_path.read_text(encoding='utf-8')).get('status', 'INVALID') if direct_review_path.is_file() else 'NOT_SEPARATE'
    reporting_status = summary.get('reporting_status', 'INVALID')
    eligible = reporting_status == 'COMPLETE' and publication_marker in {'FROZEN', 'COMPLETED'}
    if eligible:
        summaries.append(summary)
    readiness_rows.append({'model_key': model_key, 'reporting_summary': str(configured), 'reporting_status': reporting_status, 'publication_marker': publication_marker, 'separate_direct_review': separate_direct_review, 'eligible': eligible})

readiness = pd.DataFrame(readiness_rows)
display(readiness)

,model_key,reporting_summary,reporting_status,publication_marker,separate_direct_review,eligible
0,lr,artifacts/dataset_b/lr/final/lr_final_15f_2026...,COMPLETE,FROZEN,NOT_SEPARATE,True
1,rf,artifacts/dataset_b/rf/final/rf_final_15f_2026...,COMPLETE,FROZEN,NOT_SEPARATE,True
2,xgboost,artifacts/dataset_b/xgboost/final/xgb_final_15...,COMPLETE,COMPLETED,NOT_SEPARATE,True
3,plain_mlp,artifacts/dataset_b/plain_mlp/final/plain_mlp_...,COMPLETE,FROZEN,NOT_SEPARATE,True
4,realmlp,artifacts/dataset_b/realmlp/final/realmlp_fina...,COMPLETE,FROZEN,PASS,True
5,tabmmini,artifacts/dataset_b/tabmmini/final/tabmmini_fi...,COMPLETE,FROZEN,PASS,True


## Results

Cross-model tables và charts chỉ được mở sau khi gate dưới đây PASS.

In [3]:
comparison_ready = False
if len(summaries) == len(FINAL_MODEL_KEYS):
    validate_cross_model_summaries(summaries)
    comparison_ready = True

if comparison_ready:
    display(Markdown('**PASS:** Sáu final reporting packages đã sẵn sàng cho tổng hợp.'))
else:
    display(Markdown(f'**PENDING:** mới có {len(summaries)}/{len(FINAL_MODEL_KEYS)} final reporting packages đủ điều kiện; chưa mở so sánh.'))

**PASS:** Sáu final reporting packages đã sẵn sàng cho tổng hợp.

### Visual contract sau khi gate PASS

1. CV ROC-AUC: fold dot plot theo model, cùng scale.
2. RQ1 metrics: small-multiple forest plots với point estimate và CI 95%.
3. Calibration: small multiples cùng trục và đường lý tưởng.
4. Confusion/recall: bảng và heatmap chuẩn hóa.
5. Threshold và timing: bảng; chỉ so sánh timing khi định nghĩa tương đương.

Không dùng radar chart, không xếp hạng bằng metric không cùng hướng và không suy ra statistical significance từ CI riêng lẻ.

## Takeaways

Cả sáu model đã được nối vào đúng final reporting packages và gate lineage/readiness đã PASS. Notebook này chỉ xác nhận đầu vào tổng hợp đã đầy đủ; RQ2/RQ3 vẫn tuân theo các quyết định prerequisite riêng trong handbook.